<a href="https://colab.research.google.com/github/taifl2/hadaf-certifications-agent/blob/main/Hadaf_AI_Certifications_Advisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Running locally instead of Colab? Install once with:
# pip install -r requirements.txt

In [ ]:
# Only the packages this notebook actually imports
!pip install -q -U langchain-openai langchain-community langgraph tavily-python

In [ ]:
import os
import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("أدخلي مفتاح OpenAI API: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("أدخلي مفتاح Tavily API: ")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

# أداة البحث المباشر في الويب
search_tool = TavilySearchResults(
    max_results=5,
    description="تُستخدم للبحث في موقع صندوق هدف (hrdf.org.sa) والمواقع الرسمية عن الشهادات المعتمدة ورسومها."
)

# نموذج المحادثة
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# التوجيه لشخصية الوكيل وترتيب الإجابة
system_instructions = """
أنت مستشار مهني خبير في برنامج دعم الشهادات المهنية الاحترافية من صندوق تنمية الموارد البشرية (هدف).

عندما يسألك المستخدم عن أي مجال (مثل: الذكاء الاصطناعي، الأمن السيبراني، تحليل البيانات، إلخ):
1. ابحث في الإنترنت عن أفضل الشهادات العالمية المعتمدة في هذا المجال والمدعومة من "هدف".
2. رتّب إجابتك بالنقاط التالية حرفياً:
   - اسم الشهادة الدولية والجهة المانحة.
   - مميزات الشهادة وأهميتها في سوق العمل.
   - حالة الدعم من هدف ورسوم الاختبار ومبلغ التعويض بالريال.
   - طريقة التسجيل وطلب التعويض عبر بوابة هدف.
   - رابط صفحة منتج الشهادات المهنية الاحترافية على موقع هدف:
     https://www.hrdf.org.sa/products-and-services/programs/individuals/training/professional-certificates/
"""

# بناء الوكيل عبر LangGraph
hadaf_agent = create_react_agent(
    model=llm,
    tools=[search_tool],
    prompt=system_instructions
)

print(" الوكيل جاهز للعمل!")

In [ ]:
from langchain_core.messages import HumanMessage

question = "وش أفضل شهادة للذكاء الاصطناعي ؟"

response = hadaf_agent.invoke({
    "messages": [HumanMessage(content=question)]
})

print(response["messages"][-1].content)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# تفعيل الذاكرة داخل الوكيل
memory = MemorySaver()

hadaf_agent = create_react_agent(
    model=llm,
    tools=[search_tool],
    prompt=system_instructions,
    checkpointer=memory  # هنا الذاكرة
)

# عند السؤال نمرر session_id خاص بالمستخدم
config = {"configurable": {"thread_id": "user_session_1"}}

# السؤال الأول
resp1 = hadaf_agent.invoke({"messages": [HumanMessage(content="وش أفضل شهادة للذكاء الاصطناعي؟")]}, config=config)
print(resp1["messages"][-1].content)

# السؤال الثاني (سيعرف الشهادة تلقائياً من الذاكرة!)
resp2 = hadaf_agent.invoke({"messages": [HumanMessage(content="هل تناسب شخص ما عنده خبرة برمجية؟")]}, config=config)
print(resp2["messages"][-1].content)

In [ ]:
# سؤال ثالث: الاستفسار عن التعويض لنفس الشهادة المذكورة
resp3 = hadaf_agent.invoke(
    {"messages": [HumanMessage(content="كم بالضبط يعوضني هدف عنها بالريال وكيف أرفع الفاتورة؟")]},
    config=config
)
print(resp3["messages"][-1].content)